In [1]:
print(1)

1


In [2]:
# 50	Process a file too big for memory (generators/chunking)	Google-reported (discussion)	Streaming, memory awareness

In [ ]:
# #chunk
# #memoey
# #data


# Streaming

# Generators

# Chunking

# Memory awareness

In [3]:
# 100 GB CSV / log file / flat file

In [4]:
# 8 GB / 16 GB / 32 GB

In [5]:
# How do you process it safely?

In [7]:
# Bad approach

with open("sales.csv", "r") as file:
    data = file.readlines()
    print(data)

['user_id,name,revenue\n', '1,Amit,500\n', '2,Prem,900\n', '3,Sara,700\n', '4,John,300\n', '5,Nina,1200\n']


In [9]:
# 1. Best basic approach: read line by line

def process_large_file(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            # process one line at a time
            print(line)
            
process_large_file("sales.csv")
# Time complexity: O(n)
# Space complexity: O(1)

user_id,name,revenue
1,Amit,500
2,Prem,900
3,Sara,700
4,John,300
5,Nina,1200


In [11]:
# Only one line is in memory at a time.?

In [14]:
# 2. Generator approach

def read_large_file(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            yield line.strip()


for line in read_large_file("sales.csv"):
    print(line)

user_id,name,revenue
1,Amit,500
2,Prem,900
3,Sara,700
4,John,300
5,Nina,1200


In [15]:
def numbers():
    yield 1
    yield 2
    yield 3


for value in numbers():
    print(value)

1
2
3


In [16]:
def numbers():
    yield 1
    yield 2
    yield 3


numbers()

<generator object numbers at 0x10c9b0430>

In [18]:
def count_lines(file_path):
    count = 0

    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            count += 1

    return count


print(count_lines("sales.csv"))

6


In [20]:
import csv

def sum_column(file_path, column_name):
    total = 0.0

    with open(file_path, "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)

        for row in reader:
            try:
                total += float(row[column_name])
            except ValueError:
                continue

    return total


print(sum_column("sales.csv", "revenue"))

3600.0


In [21]:
import csv
import heapq

def top_n_by_column(file_path, column_name, n):
    heap = []

    with open(file_path, "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)

        for row_number, row in enumerate(reader, start=2):
            try:
                value = float(row[column_name])
            except ValueError:
                continue

            row[column_name] = value

            heapq.heappush(heap, (value, row_number, row))

            if len(heap) > n:
                heapq.heappop(heap)

    result = [item[2] for item in heap]
    result.sort(key=lambda row: row[column_name], reverse=True)

    return result


top_rows = top_n_by_column("sales.csv", "revenue", 10)

for row in top_rows:
    print(row)

{'user_id': '5', 'name': 'Nina', 'revenue': 1200.0}
{'user_id': '2', 'name': 'Prem', 'revenue': 900.0}
{'user_id': '3', 'name': 'Sara', 'revenue': 700.0}
{'user_id': '1', 'name': 'Amit', 'revenue': 500.0}
{'user_id': '4', 'name': 'John', 'revenue': 300.0}


In [22]:
# 6. Chunking approach

# Sometimes you want to process fixed-size chunks.

# Example:
def read_in_chunks(file_path, chunk_size=10000):
    chunk = []

    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            chunk.append(line.strip())

            if len(chunk) == chunk_size:
                yield chunk
                chunk = []

        if chunk:
            yield chunk


for chunk in read_in_chunks("sales.csv", chunk_size=1):
    print("Processing chunk size:", len(chunk))

    # process this chunk
    # write result somewhere


Processing chunk size: 1
Processing chunk size: 1
Processing chunk size: 1
Processing chunk size: 1
Processing chunk size: 1
Processing chunk size: 1


In [23]:
import csv
import heapq
from collections import defaultdict

def create_sample_csv(file_path):
    rows = [
        {"order_id": "1", "customer_id": "C001", "amount": "100"},
        {"order_id": "2", "customer_id": "C002", "amount": "250"},
        {"order_id": "3", "customer_id": "C001", "amount": "300"},
        {"order_id": "4", "customer_id": "C003", "amount": "50"},
        {"order_id": "5", "customer_id": "C002", "amount": "700"},
    ]

    with open(file_path, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=["order_id", "customer_id", "amount"]
        )
        writer.writeheader()
        writer.writerows(rows)


def stream_rows(file_path):
    with open(file_path, "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)

        for row in reader:
            yield row


def total_amount_by_customer(file_path):
    customer_total = defaultdict(float)

    for row in stream_rows(file_path):
        try:
            amount = float(row["amount"])
        except ValueError:
            continue

        customer_total[row["customer_id"]] += amount

    return dict(customer_total)


def top_n_orders(file_path, n):
    heap = []

    for row_number, row in enumerate(stream_rows(file_path), start=2):
        try:
            amount = float(row["amount"])
        except ValueError:
            continue

        row["amount"] = amount

        heapq.heappush(heap, (amount, row_number, row))

        if len(heap) > n:
            heapq.heappop(heap)

    result = [item[2] for item in heap]
    result.sort(key=lambda row: row["amount"], reverse=True)

    return result


file_path = "orders.csv"
create_sample_csv(file_path)

print("Total amount by customer:")
print(total_amount_by_customer(file_path))

print("Top 2 orders:")
print(top_n_orders(file_path, 2))

Total amount by customer:
{'C001': 400.0, 'C002': 950.0, 'C003': 50.0}
Top 2 orders:
[{'order_id': '5', 'customer_id': 'C002', 'amount': 700.0}, {'order_id': '3', 'customer_id': 'C001', 'amount': 300.0}]


In [24]:
# 49	Validate & convert timestamps between formats; compute deltas	Common DE	Time handling

In [25]:
# Event logs
# Payment events
# Order timestamps
# SLA calculation
# Session duration
# Pipeline latency
# Timezone conversion

In [26]:
# "2026-06-17 10:30:00"
# "17/06/2026 10:30 AM"
# "2026-06-17T03:30:00Z"

In [27]:
from datetime import datetime

def parse_timestamp(timestamp_str, input_format):
    try:
        return datetime.strptime(timestamp_str, input_format)
    except ValueError:
        return None


timestamp = "2026-06-17 10:30:00"
fmt = "%Y-%m-%d %H:%M:%S"

dt = parse_timestamp(timestamp, fmt)

if dt is None:
    print("Invalid timestamp")
else:
    print("Valid timestamp:", dt)

Valid timestamp: 2026-06-17 10:30:00


In [29]:
# %Y = 4-digit year        2026
# %m = month              06
# %d = day                17
# %H = hour 24 format     10
# %I = hour 12 format     10
# %M = minute             30
# %S = second             00
# %p = AM/PM              AM

In [31]:
from datetime import datetime

def validate_and_convert(timestamp_str, input_format, output_format="%Y-%m-%d %H:%M:%S"):
    if timestamp_str is None:
        return None

    timestamp_str = timestamp_str.strip()

    if timestamp_str == "":
        return None

    try:
        dt = datetime.strptime(timestamp_str, input_format)
        return dt.strftime(output_format)
    except ValueError:
        return None


print(validate_and_convert("17/06/2026 10:30 AM", "%d/%m/%Y %I:%M %p"))
print(validate_and_convert("wrong-date", "%d/%m/%Y %I:%M %p"))
print(validate_and_convert("", "%d/%m/%Y %I:%M %p"))

2026-06-17 10:30:00
None
None


In [32]:
from datetime import datetime

def parse_multiple_formats(timestamp_str):
    formats = [
        "%Y-%m-%d %H:%M:%S",
        "%d/%m/%Y %I:%M %p",
        "%Y-%m-%dT%H:%M:%SZ",
        "%Y/%m/%d %H:%M:%S",
    ]

    for fmt in formats:
        try:
            return datetime.strptime(timestamp_str, fmt)
        except ValueError:
            continue

    return None


timestamps = [
    "2026-06-17 10:30:00",
    "17/06/2026 10:30 AM",
    "2026-06-17T03:30:00Z",
    "bad-date"
]

for ts in timestamps:
    dt = parse_multiple_formats(ts)

    if dt:
        print(ts, "=>", dt.strftime("%Y-%m-%d %H:%M:%S"))
    else:
        print(ts, "=> invalid")

2026-06-17 10:30:00 => 2026-06-17 10:30:00
17/06/2026 10:30 AM => 2026-06-17 10:30:00
2026-06-17T03:30:00Z => 2026-06-17 03:30:00
bad-date => invalid


In [34]:
from datetime import datetime

def parse_iso_utc(ts):
    return datetime.fromisoformat(ts.replace("Z", "+00:00"))


def payment_settlement_delta(payment):
    created = parse_iso_utc(payment["payment_created_time"])
    settled = parse_iso_utc(payment["payment_settled_time"])

    delta = settled - created

    return {
        "payment_id": payment["payment_id"],
        "settlement_seconds": delta.total_seconds(),
        "settlement_minutes": delta.total_seconds() / 60
    }


payment = {
    "payment_id": "P001",
    "payment_created_time": "2026-06-17T03:00:00Z",
    "payment_settled_time": "2026-06-17T03:05:30Z"
}

print(payment_settlement_delta(payment))

{'payment_id': 'P001', 'settlement_seconds': 330.0, 'settlement_minutes': 5.5}


In [37]:
# 48	Two sorted lists → merge / dedup preserving order	Common DE	Merge logic

In [38]:
# Merge two sorted files
# Merge two sorted event streams
# Merge customer IDs from two sources
# Remove duplicates while preserving sorted order

In [39]:
list1 = [1, 2, 3, 5, 7]
list2 = [2, 3, 4, 6, 7]

In [40]:
[1, 2, 3, 4, 5, 6, 7]

[1, 2, 3, 4, 5, 6, 7]

In [41]:
list1+list2

[1, 2, 3, 5, 7, 2, 3, 4, 6, 7]

In [46]:
sorted(set(list1+list2))

[1, 2, 3, 4, 5, 6, 7]

In [47]:
def merge_dedup_sorted(list1, list2):
    return sorted(set(list1 + list2))


list1 = [1, 2, 3, 5, 7]
list2 = [2, 3, 4, 6, 7]

print(merge_dedup_sorted(list1, list2))

[1, 2, 3, 4, 5, 6, 7]


In [48]:
# Time complexity: O((m + n) log(m + n))
# Space complexity: O(m + n)

In [49]:
def merge_dedup_sorted(list1, list2):
    i = 0
    j = 0
    result = []

    def add_if_new(value):
        if not result or result[-1] != value:
            result.append(value)

    while i < len(list1) and j < len(list2):
        if list1[i] < list2[j]:
            add_if_new(list1[i])
            i += 1

        elif list1[i] > list2[j]:
            add_if_new(list2[j])
            j += 1

        else:
            add_if_new(list1[i])
            i += 1
            j += 1

    while i < len(list1):
        add_if_new(list1[i])
        i += 1

    while j < len(list2):
        add_if_new(list2[j])
        j += 1

    return result


list1 = [1, 2, 3, 5, 7]
list2 = [2, 3, 4, 6, 7]

print(merge_dedup_sorted(list1, list2))

[1, 2, 3, 4, 5, 6, 7]


In [51]:
def merge_sorted(list1, list2):
    i = 0
    j = 0
    result = []

    while i < len(list1) and j < len(list2):
        if list1[i] <= list2[j]:
            result.append(list1[i])
            i += 1
        else:
            result.append(list2[j])
            j += 1
    print(result)
    result.extend(list1[i:])
    result.extend(list2[j:])

    return result


print(merge_sorted([1, 3, 5], [2, 4, 6]))

[1, 2, 3, 4, 5]
[1, 2, 3, 4, 5, 6]


In [52]:
def merge_dedup_generator(iter1, iter2):
    sentinel = object()

    a = next(iter1, sentinel)
    b = next(iter2, sentinel)
    last_output = sentinel

    while a is not sentinel and b is not sentinel:
        if a < b:
            value = a
            a = next(iter1, sentinel)
        elif a > b:
            value = b
            b = next(iter2, sentinel)
        else:
            value = a
            a = next(iter1, sentinel)
            b = next(iter2, sentinel)

        if last_output is sentinel or value != last_output:
            yield value
            last_output = value

    while a is not sentinel:
        if last_output is sentinel or a != last_output:
            yield a
            last_output = a
        a = next(iter1, sentinel)

    while b is not sentinel:
        if last_output is sentinel or b != last_output:
            yield b
            last_output = b
        b = next(iter2, sentinel)


list1 = [1, 2, 3, 5, 7]
list2 = [2, 3, 4, 6, 7]

result = list(merge_dedup_generator(iter(list1), iter(list2)))
print(result)

[1, 2, 3, 4, 5, 6, 7]


In [53]:
def read_sorted_int_file(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()

            if line:
                yield int(line)


def write_merged_dedup_file(file1, file2, output_file):
    stream1 = read_sorted_int_file(file1)
    stream2 = read_sorted_int_file(file2)

    with open(output_file, "w", encoding="utf-8") as out:
        for value in merge_dedup_generator(stream1, stream2):
            out.write(str(value) + "\n")

In [55]:
# 47	Sum values in a range A..B (clarify: ints? list? repeated queries?)	Google-reported	Tests whether you CLARIFY before coding?

In [56]:
# Google-reported

# Tests whether you clarify before coding

# Range sum

# Prefix sum

# Repeated queries

In [58]:
# 1. Do you mean sum of integers from A to B?

#    Example: A=3, B=7 → 3+4+5+6+7

# 2. Or do you mean sum of values in a list between index A and index B?

#    Example: nums=[2,4,6,8], A=1, B=3 → 4+6+8

# 3. Is A..B inclusive or exclusive?

# 4. Will there be only one query or many repeated queries?

# 5. Can values be negative?

# 6. Can the array be updated between queries?

In [60]:
def sum_range_integers(a, b):
    total = 0

    for num in range(a, b + 1):
        total =total + num

    return total


print(sum_range_integers(3, 7))

25


In [61]:
def sum_1_to_n(n):
    return n * (n + 1) // 2


def sum_range_integers(a, b):
    if a > b:
        a, b = b, a

    return sum_1_to_n(b) - sum_1_to_n(a - 1)


print(sum_range_integers(3, 7))

25


In [62]:
def range_sum(nums, left, right):
    total = 0

    for i in range(left, right + 1):
        total += nums[i]

    return total


nums = [2, 4, 6, 8, 10]

print(range_sum(nums, 1, 3))

18


In [63]:
class RangeSum:
    def __init__(self, nums):
        self.prefix = [0]

        for num in nums:
            self.prefix.append(self.prefix[-1] + num)

    def query(self, left, right):
        return self.prefix[right + 1] - self.prefix[left]


nums = [2, 4, 6, 8, 10]

rs = RangeSum(nums)

print(rs.query(1, 3))
print(rs.query(0, 4))
print(rs.query(2, 2))

18
30
6


In [64]:
class FenwickTree:
    def __init__(self, nums):
        self.n = len(nums)
        self.tree = [0] * (self.n + 1)
        self.nums = nums[:]

        for i, num in enumerate(nums):
            self._add(i + 1, num)

    def _add(self, index, delta):
        while index <= self.n:
            self.tree[index] += delta
            index += index & -index

    def update(self, index, new_value):
        delta = new_value - self.nums[index]
        self.nums[index] = new_value
        self._add(index + 1, delta)

    def prefix_sum(self, index):
        total = 0
        index += 1

        while index > 0:
            total += self.tree[index]
            index -= index & -index

        return total

    def range_sum(self, left, right):
        if left == 0:
            return self.prefix_sum(right)

        return self.prefix_sum(right) - self.prefix_sum(left - 1)


nums = [2, 4, 6, 8, 10]

ft = FenwickTree(nums)

print(ft.range_sum(1, 3))  # 4 + 6 + 8 = 18

ft.update(2, 100)          # nums[2] changes from 6 to 100

print(ft.range_sum(1, 3))  # 4 + 100 + 8 = 112

18
112


In [66]:
class RangeSum:
    def __init__(self, nums):
        self.prefix = [0]

        for num in nums:
            self.prefix.append(self.prefix[-1] + num)

    def query(self, left, right):
        return self.prefix[right + 1] - self.prefix[left]

nums = [2, 4, 6, 8, 10]
rs = RangeSum(nums)
print(rs.query(1, 3))

18


In [68]:
# Mini GROUP BY in Python

# Replicates SQL aggregation

# HashMap / defaultdict

# Data Engineering common task

In [70]:
# CSV rows

# JSON API records

# Log events

# Payment transactions

# Order lines

# Customer events

In [71]:
def group_sum(rows, group_key, value_key):
    result = {}

    for row in rows:
        key = row[group_key]
        value = row[value_key]

        if key not in result:
            result[key] = 0

        result[key] += value

    return result


rows = [
    {"customer_id": "C001", "amount": 100},
    {"customer_id": "C002", "amount": 250},
    {"customer_id": "C001", "amount": 300},
    {"customer_id": "C003", "amount": 50},
    {"customer_id": "C002", "amount": 700},
]

print(group_sum(rows, "customer_id", "amount"))

{'C001': 400, 'C002': 950, 'C003': 50}


In [73]:
def count_letters(text):
    freq = {}
    for char in text:
        if char not in freq:
            freq[char] = 0
        freq[char] += 1
    return freq
text = "google"
print(count_letters(text))

{'g': 2, 'o': 2, 'l': 1, 'e': 1}


In [74]:
def count_letters(text):
    freq = {}

    for char in text:
        if char not in freq:
            freq[char] = 0

        freq[char] += 1

    return freq


text = "google"
print(count_letters(text))

{'g': 2, 'o': 2, 'l': 1, 'e': 1}


In [75]:
from collections import Counter

def count_by_key(rows, key):
    freq = Counter()
    for row in rows:
        freq[row[key]] += 1
    return dict(freq)

    
events = [
    {"event_type": "click"},
    {"event_type": "view"},
    {"event_type": "click"},
    {"event_type": "purchase"},
    {"event_type": "view"},
    {"event_type": "click"},
]
print(count_by_key(events, "event_type"))

{'click': 3, 'view': 2, 'purchase': 1}


In [77]:
from collections import Counter
def word_frequency_sorted(text):

    words = text.lower().split()

    freq = Counter(words)

    result = sorted(

        freq.items(),

        key=lambda item: (-item[1], item[0])

    )

    return result

text = "sql python data sql python google"

print(word_frequency_sorted(text))

[('python', 2), ('sql', 2), ('data', 1), ('google', 1)]


In [79]:
4	Uncommon words between two sentences	Google-reported	Counter across two inputs?

Object `inputs` not found.


In [80]:
s1 = "this apple is sweet"
s2 = "this apple is sour"


In [108]:
def un_common(s1,s2):
    d={}
    result=[]
    s1 = s1.split()
    s2 = s2.split()
    for s in s1:
        if s in d:
            d[s] = +1
        else:
            d[s] = 1
    print(d)
    for s in s2:
        if s in d:
            d[s] =d[s]- 1
        else:
            d[s] = 1

    for i, row in d.items():
        if row>0:
            result.append(i)
            
        print(i, row)

    
    return result       

In [109]:
un_common(s1,s2)

{'this': 1, 'apple': 1, 'is': 1, 'sweet': 1}
this 0
apple 0
is 0
sweet 1
sour 1


['sweet', 'sour']

In [112]:
def un_common(s1, s2):
    d = {}
    result = []

    words = s1.split() + s2.split()

    for word in words:
        if word in d:
            d[word] += 1
        else:
            d[word] = 1

    print(d)
    for word, count in d.items():
        if count == 1:
            result.append(word)

    return result


print(un_common("this apple is sweet", "this apple is sour"))
print(un_common("apple apple", "banana banana"))

{'this': 2, 'apple': 2, 'is': 2, 'sweet': 1, 'sour': 1}
['sweet', 'sour']
{'apple': 2, 'banana': 2}
[]


In [113]:
# 43	Print key of nth-highest value in a dict (tie-break by key sort)	Google-reported	Dict sorting + edge cases

In [114]:
43	Print key of nth-highest value in a dict (tie-break by key sort)	Google-reported	Dict sorting + edge cases?

Object `cases` not found.


In [115]:
scores = {
    "alice": 90,
    "bob": 95,
    "charlie": 95,
    "david": 80
}

In [117]:
# bob      95
# charlie  95
# alice    90
# david    80

In [124]:
sorted(scores.items(),key=lambda item: (-item[1], item[0]))

[('bob', 95), ('charlie', 95), ('alice', 90), ('david', 80)]

In [121]:
def nth_highest_key(data, n):
    sorted_items = sorted(
        data.items(),
        key=lambda item: (-item[1], item[0])
    )

    return sorted_items[n - 1][0]


scores = {
    "alice": 90,
    "bob": 95,
    "charlie": 95,
    "david": 80
}

print(nth_highest_key(scores, 1))
print(nth_highest_key(scores, 2))
print(nth_highest_key(scores, 3))

bob
charlie
alice


In [126]:
scores.items()

dict_items([('alice', 90), ('bob', 95), ('charlie', 95), ('david', 80)])

In [128]:
def nth_highest_key(data, n):
    if not data or n <= 0 or n > len(data):
        return None

    sorted_items = sorted(
        data.items(),
        key=lambda item: (-item[1], item[0])
    )
    print(sorted_items)
    return sorted_items[n - 1][0]


scores = {
    "alice": 90,
    "bob": 95,
    "charlie": 95,
    "david": 80
}

print(nth_highest_key(scores, 1))
print(nth_highest_key(scores, 2))
print(nth_highest_key(scores, 3))

[('bob', 95), ('charlie', 95), ('alice', 90), ('david', 80)]
bob
[('bob', 95), ('charlie', 95), ('alice', 90), ('david', 80)]
charlie
[('bob', 95), ('charlie', 95), ('alice', 90), ('david', 80)]
alice


In [129]:
# 1	Forward-fill: replace None with previous non-null value	Google-reported	Data cleaning (handles leading None, empty, None input)